<a href="https://colab.research.google.com/github/Akhila-010/GenAI_Tasks/blob/main/News_and_Emails.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q openai beautifulsoup4 requests

In [ ]:
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
from getpass import getpass

In [ ]:
api_key = getpass("Enter your OpenAI API Key: ")

client = OpenAI(
    api_key=api_key
)

Enter your OpenAI API Key: ··········


In [ ]:
# FUNCTION TO SCRAPE NEWS

import requests
from bs4 import BeautifulSoup

def scrape_news():

    url = "https://www.thehindu.com/"

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }

    try:

        response = requests.get(
            url,
            headers=headers,
            timeout=10
        )

        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        headlines = []

        # Extract headlines
        for item in soup.find_all(["h1", "h2", "h3"]):

            text = item.get_text(strip=True)

            if len(text) > 30:
                headlines.append(text)

        # Remove duplicates
        headlines = list(set(headlines))

        return headlines[:10]

    except Exception as e:

        print("Error:", e)

        return []

# # TEST
# news = scrape_news()

# for i, headline in enumerate(news, 1):
#     print(f"{i}. {headline}")

In [ ]:
# FUNCTION TO SCRAPE EMAILS
import re

def scrape_emails(url):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/120.0.0.0 Safari/537.36"
        )
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b'
        emails = re.findall(email_pattern, response.text)
        return list(set(emails))
    except requests.exceptions.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return []
    except Exception as e:
        print(f"An error occurred during email scraping: {e}")
        return []


In [ ]:
# FUNCTION TO SUMMARIZE NEWS

def summarize_news(headlines):

    # Convert list into text
    news_text = "\n".join(headlines)

    # AI Prompt
    prompt = f"""
    Summarize these news headlines into a short daily news report.

    Headlines:
    {news_text}
    """

    try:

        # OpenAI API call
        response = client.chat.completions.create(

            model="gpt-4o-mini",

            messages=[

                {
                    "role": "system",
                    "content": "You are a professional news reporter."
                },

                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        summary = response.choices[0].message.content

        return summary

    except Exception as e:

        print("Error while generating summary:")
        print(e)

        return "Summary generation failed."

In [ ]:
# FUNCTION TO SAVE NEWS
def save_report(headlines, summary):

    with open("daily_news_report.txt", "w", encoding="utf-8") as file:

        file.write("=========== TOP HEADLINES ===========\n\n")

        for i, news in enumerate(headlines, 1):
            file.write(f"{i}. {news}\n\n")

        file.write("\n=========== AI SUMMARY ===========\n\n")

        file.write(summary)

    print("\nNews report saved as daily_news_report.txt")

In [ ]:
#main Program

print("Fetching latest news...\n")

headlines = scrape_news()

# Check headlines
if len(headlines) == 0:

    print("No headlines found.")

else:

    print("=========== TOP HEADLINES ===========\n")

    for i, news in enumerate(headlines, 1):

        print(f"{i}. {news}\n")

    print("\nGenerating AI Summary...\n")

    summary = summarize_news(headlines)

    print("=========== DAILY NEWS REPORT ===========\n")

    print(summary)

    # Save report
    save_report(headlines, summary)

Fetching latest news...

=========== TOP HEADLINES ===========

1. Tamil Nadu government formation: DMK rejects overtures from AIADMK, sources say

2. Urrak plantation hopping in Goa

3. 65% of newly elected Bengal MLAs face criminal cases; 61% are ‘crorepatis’: ADR

4. Shakespeare’s food universe: Eat the Bard’s words on his birthday

5. Rampant RCB faces a tricky LSG challenge

6. Statehood will be achieved soon: Puducherry CM Rangasamy

7. Assam Congress to analyse poll defeat

8. Homeschooled Suhail transforms robotics landscape through self-learning

9. On Mother’s Day | Green Humour by Rohan Chakravarty

10. Explained: Rise of BJP in West Bengal


Generating AI Summary...

=========== DAILY NEWS REPORT ===========

In today’s news report, the Tamil Nadu government formation has hit a snag as the DMK has reportedly declined overtures from the AIADMK for collaboration. Meanwhile, in Goa, residents and tourists engage in a popular practice known as Urrak plantation hopping. In West 

In [ ]:
# Example usage of email scraping
# You can modify the URL to a page where you expect to find emails
# The previous URL 'https://www.thehindu.com/contact-us/' resulted in a 404 error.
# Let's try a different, more general page or a known working contact page if available.
# Please replace 'https://example.com/some-page-with-emails' with an actual URL if needed.
email_scrape_url = "https://www.thehindu.com/"
print(f"\nFetching emails from {email_scrape_url}...")
email_addresses = scrape_emails(email_scrape_url)

if email_addresses:
    print("\n=========== EXTRACTED EMAIL ADDRESSES ===========\n")
    for email in email_addresses:
        print(email)
    # Optionally, save emails to a file or integrate with an email sending service
    with open("extracted_emails.txt", "w") as f:
        for email in email_addresses:
            f.write(f"{email}\n")
    print("\nEmail addresses saved to extracted_emails.txt")
else:
    print("No email addresses found on the specified page.")


# To set up daily updates, you would typically use a scheduling tool outside of Colab,
# such as cron jobs on a server, or cloud-based schedulers like Google Cloud Scheduler
# combined with services like Cloud Functions or Cloud Run to execute this notebook daily.
# Within Python, you could use libraries like 'schedule' for simple in-script scheduling,
# but it requires the script to be continuously running.


Fetching emails from https://www.thehindu.com/...

=========== EXTRACTED EMAIL ADDRESSES ===========

customersupport@thehindu.co.in

Email addresses saved to extracted_emails.txt
